In [1]:
import os
from dotenv import load_dotenv

load_dotenv()

GROQ_API_KEY = os.getenv("GROQ_API_KEY")

In [2]:
from typing_extensions import TypedDict
from langgraph.graph import StateGraph, END, START
from typing import Annotated
from langgraph.graph.message import add_messages

In [3]:
class State(TypedDict):
    messages: Annotated[list, add_messages]

In [4]:
from langchain_groq import ChatGroq
llm = ChatGroq(model="qwen/qwen3-32b", groq_api_key=GROQ_API_KEY)
llm

ChatGroq(output_version=None, profile={'max_input_tokens': 131072, 'max_output_tokens': 16384, 'image_inputs': False, 'audio_inputs': False, 'video_inputs': False, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': True, 'tool_calling': True}, client=<groq.resources.chat.completions.Completions object at 0x000001EBCA2D2E10>, async_client=<groq.resources.chat.completions.AsyncCompletions object at 0x000001EBCA299AF0>, model_name='qwen/qwen3-32b', model_kwargs={}, groq_api_key=SecretStr('**********'), groq_api_base=None, groq_proxy=None)

In [5]:
def superbot(state: State):
    return {"messages" : [llm.invoke(state["messages"])]}

In [6]:
graph_builder = StateGraph(State)

graph_builder.add_node("superbot", superbot)

graph_builder.add_edge(START, "superbot")

graph_builder.add_edge("superbot", END)

In [7]:
graph = graph_builder.compile()

In [9]:
graph.invoke({"messages" : "Hi, my name is Prashant and I like Cricket"})

{'messages': [HumanMessage(content='Hi, my name is Prashant and I like Cricket', additional_kwargs={}, response_metadata={}, id='ceadd999-cc6e-4fd6-bba9-88509f0dff54'),
  AIMessage(content='<think>\nOkay, the user introduced himself as Prashant and mentioned that he likes cricket. Let me start by acknowledging his introduction and showing interest in his hobby. I should respond in a friendly and engaging manner. Maybe ask him about his favorite teams or players to keep the conversation going. I can also share some general info about cricket or ask if he plays as well. Let me make sure the tone is positive and encouraging to foster further interaction.\n</think>\n\nHello Prashant! Nice to meet you! 🏏 Cricket is such an amazing sport—do you follow any particular teams or players closely? Also, are you someone who plays, or are you more of a passionate spectator? Let me know—I’d love to hear more about what you enjoy most about the game!', additional_kwargs={}, response_metadata={'token_u

### Stream the response

In [10]:
graph.stream({"messages": "Hello, my name is Prashant"})

<generator object Pregel.stream at 0x000001EBC9225780>

In [12]:
for event in graph.stream({"messages": "Hello, my name is Prashant"}, stream_mode="values"):
    print(event)

{'messages': [HumanMessage(content='Hello, my name is Prashant', additional_kwargs={}, response_metadata={}, id='c293fb3f-cb60-4192-9558-fe254ad6332b')]}
{'messages': [HumanMessage(content='Hello, my name is Prashant', additional_kwargs={}, response_metadata={}, id='c293fb3f-cb60-4192-9558-fe254ad6332b'), AIMessage(content='<think>\nOkay, the user introduced himself as Prashant. I should respond in a friendly and welcoming manner. Let me make sure to acknowledge his name and offer assistance. I need to keep the tone positive and open-ended so he feels comfortable asking questions. Maybe add a smiley to keep it warm. Let me check for any typos and ensure the response is concise. Alright, that should do it.\n</think>\n\nHello Prashant! Nice to meet you! 😊 How can I assist you today?', additional_kwargs={}, response_metadata={'token_usage': {'completion_tokens': 105, 'prompt_tokens': 16, 'total_tokens': 121, 'completion_time': 0.189137726, 'completion_tokens_details': None, 'prompt_time':